# 🏀 OpenCommish Data Analysis Notebook

This notebook provides interactive analysis of the Yahoo NBA Fantasy Basketball data collected by OpenCommish.

## Setup

```bash
pip install -r notebooks/requirements.txt
jupyter notebook notebooks/analysis.ipynb
```

## What You'll Find Here

- **Team Overview**: Standings and roster breakdowns
- **Player Performance**: Daily stats and fantasy points
- **Bench Efficiency**: Points left on the bench vs starting lineup
- **Category Analysis**: Breakdown by stat categories (PTS, REB, AST, etc.)
- **Trends Over Time**: Multi-day performance tracking

In [ ]:
# Import libraries
import json
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from pathlib import Path

# Set up plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Configuration
DATA_DIR = Path("../data/daily_stats")
LEAGUE_ID = "93905"

print("✅ Libraries loaded successfully!")

## 1. Load and Explore Data

In [ ]:
# Find all daily stats files
stats_files = sorted(glob.glob(str(DATA_DIR / f"league_{LEAGUE_ID}_*.json")))
print(f"Found {len(stats_files)} days of data")
print(f"Date range: {Path(stats_files[0]).stem.split('_')[-1]} to {Path(stats_files[-1]).stem.split('_')[-1]}")

In [ ]:
# Load the most recent day's data
latest_file = stats_files[-1]
with open(latest_file) as f:
    latest_data = json.load(f)

print(f"📅 Date: {latest_data['date']}")
print(f"🏆 League: {latest_data['league_name']}")
print(f"📊 Week: {latest_data['week']}")
print(f"👥 Teams: {len(latest_data['teams'])}")

# Show all teams
print("\nTeams in league:")
for i, team in enumerate(latest_data['teams'], 1):
    print(f"  {i}. {team['team_name']} ({len(team['players'])} players)")

## 2. Team Performance Overview

In [ ]:
# Calculate total fantasy points per team for the latest day
team_totals = []
for team in latest_data['teams']:
    total_points = sum(p['fantasy_points'] for p in team['players'])
    active_players = [p for p in team['players'] if p['roster_position'] != 'BN']
    bench_players = [p for p in team['players'] if p['roster_position'] == 'BN']
    
    active_points = sum(p['fantasy_points'] for p in active_players)
    bench_points = sum(p['fantasy_points'] for p in bench_players)
    
    team_totals.append({
        'team_name': team['team_name'],
        'total_players': len(team['players']),
        'active_players': len(active_players),
        'bench_players': len(bench_players),
        'total_points': total_points,
        'active_points': active_points,
        'bench_points': bench_points,
        'bench_efficiency': bench_points / total_points * 100 if total_points > 0 else 0
    })

# Create DataFrame and sort by active points
df_teams = pd.DataFrame(team_totals).sort_values('active_points', ascending=False)
df_teams

In [ ]:
# Visualize team performance
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Active points by team
ax1 = axes[0, 0]
bars = ax1.barh(df_teams['team_name'], df_teams['active_points'], color='steelblue')
ax1.set_xlabel('Fantasy Points (Active Players)')
ax1.set_title('Team Performance - Active Roster')
ax1.invert_yaxis()

# Bench points
ax2 = axes[0, 1]
bars2 = ax2.barh(df_teams['team_name'], df_teams['bench_points'], color='coral')
ax2.set_xlabel('Fantasy Points (Bench Players)')
ax2.set_title('Points Left on Bench')
ax2.invert_yaxis()

# Bench efficiency %
ax3 = axes[1, 0]
bars3 = ax3.barh(df_teams['team_name'], df_teams['bench_efficiency'], color='orange')
ax3.set_xlabel('Bench Points / Total Points (%)')
ax3.set_title('Bench Efficiency (Lower is Better)')
ax3.invert_yaxis()

# Stacked bar: Active vs Bench
ax4 = axes[1, 1]
ax4.barh(df_teams['team_name'], df_teams['active_points'], label='Active', color='steelblue')
ax4.barh(df_teams['team_name'], df_teams['bench_points'], 
         left=df_teams['active_points'], label='Bench', color='coral')
ax4.set_xlabel('Total Fantasy Points')
ax4.set_title('Active vs Bench Breakdown')
ax4.legend()
ax4.invert_yaxis()

plt.tight_layout()
plt.show()

## 3. Player Performance Analysis

In [ ]:
# Collect all player data from latest day
all_players = []
for team in latest_data['teams']:
    for player in team['players']:
        player_info = {
            'name': player['name'],
            'team': team['team_name'],
            'roster_position': player['roster_position'],
            'fantasy_points': player['fantasy_points'],
            'is_bench': player['roster_position'] == 'BN'
        }
        # Add stat breakdown if available
        if player['stats']:
            for stat in player['stats']:
                stat_name = stat['display_name']
                player_info[stat_name] = stat['value']
        all_players.append(player_info)

df_players = pd.DataFrame(all_players)
print(f"Total players: {len(df_players)}")
df_players.head(10)

In [ ]:
# Top performers
print("🏆 TOP 10 PERFORMERS (All Players)")
print("=" * 60)
top_performers = df_players.nlargest(10, 'fantasy_points')[['name', 'team', 'roster_position', 'fantasy_points']]
for idx, row in top_performers.iterrows():
    print(f"{row['name']:<25} {row['team']:<20} {row['roster_position']:<6} {row['fantasy_points']:>8.2f}")

print("\n💔 TOP 5 BENCH PERFORMERS (Opportunity Cost)")
print("=" * 60)
bench_stars = df_players[df_players['is_bench']].nlargest(5, 'fantasy_points')[['name', 'team', 'fantasy_points']]
for idx, row in bench_stars.iterrows():
    print(f"{row['name']:<25} {row['team']:<20} {row['fantasy_points']:>8.2f}")

## 4. Category Breakdown

In [ ]:
# Identify stat columns (exclude metadata columns)
stat_cols = [col for col in df_players.columns 
             if col not in ['name', 'team', 'roster_position', 'fantasy_points', 'is_bench']]

if stat_cols:
    # Aggregate by team
    team_stats = df_players.groupby('team')[stat_cols].sum().sort_values('PTS', ascending=False)
    print("Team Stats by Category:")
    display(team_stats)
    
    # Visualize
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, stat in enumerate(stat_cols[:6]):  # Show first 6 stats
        ax = axes[idx]
        team_stats_sorted = team_stats.sort_values(stat, ascending=True)
        ax.barh(team_stats_sorted.index, team_stats_sorted[stat], color='steelblue')
        ax.set_xlabel(stat)
        ax.set_title(f'{stat} by Team')
    
    plt.tight_layout()
    plt.show()
else:
    print("No detailed stat categories found in data")

## 5. Multi-Day Trends

In [ ]:
# Load multiple days for trend analysis
daily_team_totals = []

for stats_file in stats_files[-7:]:  # Last 7 days
    with open(stats_file) as f:
        day_data = json.load(f)
    
    date = day_data['date']
    
    for team in day_data['teams']:
        active_players = [p for p in team['players'] if p['roster_position'] != 'BN']
        active_points = sum(p['fantasy_points'] for p in active_players)
        
        daily_team_totals.append({
            'date': date,
            'team': team['team_name'],
            'active_points': active_points
        })

df_trends = pd.DataFrame(daily_team_totals)
df_trends['date'] = pd.to_datetime(df_trends['date'])

# Plot trends
plt.figure(figsize=(12, 6))
for team in df_trends['team'].unique():
    team_data = df_trends[df_trends['team'] == team]
    plt.plot(team_data['date'], team_data['active_points'], marker='o', label=team)

plt.xlabel('Date')
plt.ylabel('Active Fantasy Points')
plt.title('Team Performance Trends (Last 7 Days)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Summary table
print("\n📊 7-Day Performance Summary")
print("=" * 60)
summary = df_trends.groupby('team')['active_points'].agg(['mean', 'min', 'max', 'std']).round(2)
summary.columns = ['Avg', 'Min', 'Max', 'StdDev']
summary.sort_values('Avg', ascending=False)

## 6. Bench Efficiency Deep Dive

In [ ]:
# Calculate opportunity cost for each team
print("🔍 BENCH EFFICIENCY ANALYSIS")
print("=" * 80)

for team in latest_data['teams']:
    team_name = team['team_name']
    players = team['players']
    
    # Separate active and bench
    active = [(p['name'], p['fantasy_points']) for p in players if p['roster_position'] != 'BN']
    bench = [(p['name'], p['fantasy_points']) for p in players if p['roster_position'] == 'BN']
    
    active_points = sum(p[1] for p in active)
    bench_points = sum(p[1] for p in bench)
    total = active_points + bench_points
    
    print(f"\n📌 {team_name}")
    print(f"   Active Points: {active_points:.2f} | Bench Points: {bench_points:.2f} | Total: {total:.2f}")
    print(f"   Bench Efficiency: {bench_points/total*100:.1f}% of points on bench")
    
    # Find best bench player
    if bench:
        best_bench = max(bench, key=lambda x: x[1])
        worst_active = min(active, key=lambda x: x[1])
        
        print(f"   💡 Best Bench: {best_bench[0]} ({best_bench[1]:.2f} pts)")
        print(f"   ⚠️  Worst Active: {worst_active[0]} ({worst_active[1]:.2f} pts)")
        
        if best_bench[1] > worst_active[1]:
            diff = best_bench[1] - worst_active[1]
            print(f"   🚨 MISOPPORTUNITY: {best_bench[0]} outscored {worst_active[0]} by {diff:.2f} points!")

---

## Next Steps

This notebook provides a foundation for analyzing your fantasy league data. Consider extending it with:

1. **Matchup Predictions**: Compare team strengths for upcoming matchups
2. **Player Consistency**: Analyze player performance variance over time
3. **Category Targeting**: Identify which categories to focus on for improvement
4. **Waiver Wire Analysis**: Compare free agents to current roster

Happy analyzing! 🏆